In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy.stats import norm

# ── Fonts ─────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "text.usetex":        False,
    "font.family":        "serif",
    "font.serif":         ["Palatino", "Palatino Linotype", "Georgia",
                           "Times New Roman", "DejaVu Serif"],
    "mathtext.fontset":   "cm",
    "axes.labelsize":     9,
    "axes.titlesize":     9,
    "xtick.labelsize":    8,
    "ytick.labelsize":    8,
    "figure.dpi":         150,
})

# ── Colour ────────────────────────────────────────────────────────────────────
TROPICAL_GREEN = (0/255, 118/255, 94/255)

def _bar_color(rgb, x, x_max=3.5, pale_mix=0.72):
    t = min(abs(x) / x_max, 1.0)
    return tuple(c + (1 - c) * pale_mix * t for c in rgb)

def _edge_color(rgb, x, x_max=3.5, pale_mix=0.35):
    t = min(abs(x) / x_max, 1.0)
    return tuple(c + (1 - c) * pale_mix * t for c in rgb)

# ── ERW simulator ─────────────────────────────────────────────────────────────
def simulate_erw_batch(n, p, n_sims, q=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    xi = np.zeros((n_sims, n + 1), dtype=np.int8)
    xi[:, 1] = np.where(rng.random(n_sims) < q, 1, -1)
    for t in range(1, n):
        idx  = rng.integers(1, t + 1, size=n_sims)
        past = xi[np.arange(n_sims), idx]
        flip = np.where(rng.random(n_sims) < p, 1, -1)
        xi[:, t + 1] = past * flip
    return xi[:, 1:].sum(axis=1).astype(float)

# ── Parameters ────────────────────────────────────────────────────────────────
P_VAL   = 0.75
N_STEPS = 50_000
N_SIMS  = 25_000
N_BINS  = 58
Q       = 0.5

rng = np.random.default_rng(271828)

raw    = simulate_erw_batch(N_STEPS, p=P_VAL, n_sims=N_SIMS, q=Q, rng=rng)
scaled = raw / np.sqrt(N_STEPS * np.log(N_STEPS))

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(2.8, 2.9))

LABEL_GREY = (0.55, 0.55, 0.55)

counts, bin_edges = np.histogram(scaled, bins=N_BINS, density=True)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_width   = bin_edges[1] - bin_edges[0]

for xc, h in zip(bin_centers, counts):
    ax.bar(xc, h, width=bin_width,
           color=_bar_color(TROPICAL_GREEN, xc),
           edgecolor=_edge_color(TROPICAL_GREEN, xc),
           linewidth=0.28, alpha=0.45, zorder=2)

x_fine = np.linspace(-4.5, 4.5, 600)
ax.plot(x_fine, norm.pdf(x_fine), color=TROPICAL_GREEN, lw=0.90, zorder=3)

ax.set_title(r"$p\!=\!0.75$", fontsize=9, pad=5, color=LABEL_GREY)
ax.set_xlabel(r"$Z_n$", labelpad=3)
ax.set_ylabel("Density", labelpad=4, fontsize=8)
ax.set_xlim(-4.5, 4.5)
ax.set_ylim(bottom=0)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_linewidth(0.55)
ax.spines["bottom"].set_linewidth(0.55)
ax.tick_params(width=0.55, length=3, direction="out")

ax.xaxis.set_major_locator(ticker.FixedLocator([-3, 0, 3]))
ax.xaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, _:
        r"$-3$" if x == -3 else (r"$3$" if x == 3 else r"$0$"))
)
ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=3))

fig.tight_layout()
print("Done.")